# Regresión Lineal para Predecir el LTV a 24 Meses

**Objetivo de aprendizaje:**
Descubrir variables que impactan el KPI de LTV a 24 meses mediante una regresión multivariable, e interpretar el modelo para apoyar la toma de decisiones.

### Contexto del caso: Tlacuachitos Express

Tlacuachitos Express es un servicio ficticio de entregas que desea predecir el valor que generará un cliente en los próximos 24 meses. El área de retención quiere utilizar este modelo para segmentar a los clientes y priorizar esfuerzos de fidelización.

### Exploración del KPI: ¿Qué es el LTV?

Discusión: ¿Qué es el Lifetime Value (LTV)? ¿Por qué es un KPI clave?

Definición del KPI: Suma de todas las compras realizadas por un cliente en sus primeros 24 meses.

## Pasos

### 1. Colección y comprensión de datos

In [1]:
import pandas as pd

df_data = pd.read_csv('data/tlacuachitos_express_customers_data.csv')
df_data.head(3)

,CustomerID,Age,Income,Tenure,Education,Industry,Geographic Location,Cohort
0,1,56,52752.677346,3,Master,Technology,Europe,2023-08-31
1,2,69,55297.364348,6,Bachelor,Technology,South America,2021-08-31
2,3,46,57978.753383,3,Bachelor,Finance,Europe,2019-05-31


In [2]:
df_transactions = pd.read_csv('data/tlacuachitos_express_transactions.csv')
df_transactions.head(3)

,CustomerID,TransactionDate,TransactionAmount
0,1,2023-08-31,524.891753
1,1,2024-10-31,794.653366
2,1,2023-12-31,223.096087


In [3]:
df_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CustomerID           1143 non-null   int64  
 1   Age                  1143 non-null   int64  
 2   Income               1143 non-null   float64
 3   Tenure               1143 non-null   int64  
 4   Education            1143 non-null   object 
 5   Industry             1143 non-null   object 
 6   Geographic Location  1143 non-null   object 
 7   Cohort               1143 non-null   object 
dtypes: float64(1), int64(3), object(4)
memory usage: 71.6+ KB


In [4]:
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31552 entries, 0 to 31551
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         31552 non-null  int64  
 1   TransactionDate    31552 non-null  object 
 2   TransactionAmount  31552 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 739.6+ KB


### 2. Limpieza de datos

In [5]:
df_transactions['TransactionDate'] = pd.to_datetime(df_transactions['TransactionDate'])
df_data['Cohort'] = pd.to_datetime(df_data['Cohort'])

In [6]:
snapshot_date = df_transactions['TransactionDate'].max()
snapshot_date

Timestamp('2025-03-31 00:00:00')

In [7]:
df_transactions['TransactionDate'].min()

Timestamp('2018-01-31 00:00:00')

In [8]:
df_data['customer_tenure'] = (snapshot_date.year - df_data['Cohort'].dt.year)*12 + (
    snapshot_date.month - df_data['Cohort'].dt.month)

df_data.head(3)

,CustomerID,Age,Income,Tenure,Education,Industry,Geographic Location,Cohort,customer_tenure
0,1,56,52752.677346,3,Master,Technology,Europe,2023-08-31,19
1,2,69,55297.364348,6,Bachelor,Technology,South America,2021-08-31,43
2,3,46,57978.753383,3,Bachelor,Finance,Europe,2019-05-31,70


In [9]:
df_master = df_transactions.merge(df_data, on='CustomerID')
df_master.head(3)

,CustomerID,TransactionDate,TransactionAmount,Age,Income,Tenure,Education,Industry,Geographic Location,Cohort,customer_tenure
0,1,2023-08-31,524.891753,56,52752.677346,3,Master,Technology,Europe,2023-08-31,19
1,1,2024-10-31,794.653366,56,52752.677346,3,Master,Technology,Europe,2023-08-31,19
2,1,2023-12-31,223.096087,56,52752.677346,3,Master,Technology,Europe,2023-08-31,19


In [10]:
df_master['customer_tenure_on_transaction'] = (
    df_master['TransactionDate'].dt.year - df_master['Cohort'].dt.year)*12 + (
    df_master['TransactionDate'].dt.month - df_master['Cohort'].dt.month
)
df_master[['Cohort', 'TransactionDate', 'customer_tenure_on_transaction']].head()

,Cohort,TransactionDate,customer_tenure_on_transaction
0,2023-08-31,2023-08-31,0
1,2023-08-31,2024-10-31,14
2,2023-08-31,2023-12-31,4
3,2023-08-31,2023-12-31,4
4,2023-08-31,2024-03-31,7


### 3. Cálculo del KPI objetivo: LTV a 24 meses

In [11]:
cltv_24_months = df_master[
    (df_master['customer_tenure'] > 24) &
    (df_master['customer_tenure_on_transaction'] <= 24)
].groupby('CustomerID')['TransactionAmount'].sum().reset_index()

cltv_24_months.rename(columns={'TransactionAmount': 'LTV'}, inplace=True)
cltv_24_months.head()

,CustomerID,LTV
0,2,11276.581901
1,3,5084.632444
2,4,3037.917187
3,5,11677.948404
4,6,4556.353725


### 4. Ingeniería de características

In [18]:
categorical_features = ['Education', 'Industry', 'Geographic Location']
numerical_features = ['Age', 'Income']

df_encoded = pd.get_dummies(df_data, columns=categorical_features, drop_first=False)

df_to_model = cltv_24_months.merge(df_encoded, on='CustomerID')

df_to_model.head()

,CustomerID,LTV,Age,Income,Tenure,Cohort,customer_tenure,Education_Bachelor,Education_High School,Education_Master,...,Industry_Education,Industry_Entertainment,Industry_Finance,Industry_Healthcare,Industry_Technology,Geographic Location_Asia,Geographic Location_Australia,Geographic Location_Europe,Geographic Location_North America,Geographic Location_South America
0,2,11276.581901,69,55297.364348,6,2021-08-31,43,True,False,False,...,False,False,False,False,True,False,False,False,False,True
1,3,5084.632444,46,57978.753383,3,2019-05-31,70,True,False,False,...,False,False,True,False,False,False,False,True,False,False
2,4,3037.917187,32,60445.266900,3,2021-02-28,49,False,True,False,...,True,False,False,False,False,False,False,False,False,True
3,5,11677.948404,60,57741.870929,5,2018-10-31,77,True,False,False,...,False,True,False,False,False,True,False,False,False,False
4,6,4556.353725,25,57132.404622,3,2022-06-30,33,False,False,True,...,False,False,False,True,False,False,False,False,True,False


In [19]:
dummy_features = (
    df_to_model.columns[
        df_to_model.columns.str.startswith(tuple(categorical_features))].values
).tolist()

### 5. Regresión lineal multivariable

In [24]:
import statsmodels.api as sm

X = df_to_model[dummy_features + numerical_features]
y = ['LTV']

X = sm.add_constant(X)

model = sm.OLS(df_to_model[y], X.astype(float))
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                    LTV   R-squared:                       0.615
Model:                            OLS   Adj. R-squared:                  0.610
Method:                 Least Squares   F-statistic:                     119.2
Date:                Fri, 28 Mar 2025   Prob (F-statistic):          1.44e-190
Time:                        18:48:48   Log-Likelihood:                -8875.8
No. Observations:                 984   AIC:                         1.778e+04
Df Residuals:                     970   BIC:                         1.785e+04
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const 

### 6. Interpretación del modelo

- ¿Qué variables son estadísticamente significativas?
- ¿Qué coeficientes son positivos o negativos?
- ¿Cómo se puede segmentar a los clientes con base en el LTV esperado?

### 7. Predicción de un nuevo cliente

In [23]:
new_customer_data = {
    'const': [1],
    'Education_Bachelor': [1],
    'Education_High School': [0],
    'Education_Master': [0],
    'Education_PhD': [0],
    'Industry_Education': [0],
    'Industry_Entertainment': [0],
    'Industry_Finance': [0],
    'Industry_Healthcare': [1],
    'Industry_Technology': [0],
    'Geographic Location_Asia': [0],
    'Geographic Location_Australia': [0],
    'Geographic Location_Europe': [0],
    'Geographic Location_North America': [0],
    'Geographic Location_South America': [1],
    'Age': [18],
    'Income': [3500],
}

new_customer_df = pd.DataFrame(new_customer_data)

predicted_ltv = results.predict(new_customer_df)

print(f"Predicted LTV for the new customer: ${predicted_ltv[0]:,.2f}")


Predicted LTV for the new customer: $1,395.20


### 8. Actividad final
Caso de aplicación:
- El equipo de marketing tiene presupuesto limitado para campañas. 
- Basado en el modelo, ¿qué tipo de cliente deberías priorizar?

